# Self-Sufficient Pipeline: ROI-Cropped → SAM-Med3D → TabPFN

This notebook implements a minimal end-to-end pipeline:
1. **Preprocess** images with ROI cropping (tumor-centered volumes)
2. **Load** SAM-Med3D model via medim
3. **Extract** embeddings from the encoder
4. **Pool** embeddings to feature vectors (Global Average Pooling)
5. **Load** labels from sheet.csv
6. **Classify** with TabPFN

## 1. Imports and Setup

In [8]:
import sys
from pathlib import Path
import numpy as np
import torch
import SimpleITK as sitk
import torchio as tio
import medim

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Define paths
SAM3D_ROOT = project_root / "SAM-Med3D-main" / "SAM-Med3D-main"
DATASET_ROOT = project_root / "data" / "gist"  # Change to your dataset
CHECKPOINT_PATH = SAM3D_ROOT / "ckpt" / "sam_med3d_turbo.pth"

# Configuration
CATEGORY = "gist"
CT_NAME = "ct_GIST_roi"  # Use _roi suffix for ROI-cropped data
TARGET_SIZE = 128
ROI_MARGIN = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Setup complete")
print(f"   Project root: {project_root}")
print(f"   SAM-Med3D root: {SAM3D_ROOT}")
print(f"   Dataset: {DATASET_ROOT}")
print(f"   Device: {DEVICE}")

✅ Setup complete
   Project root: c:\Users\cahel\Desktop\Med3Tab-PFN
   SAM-Med3D root: c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main
   Dataset: c:\Users\cahel\Desktop\Med3Tab-PFN\data\gist
   Device: cpu


## 2. Preprocessing with ROI Cropping

Preprocess images using the ROI-cropping method which:
- Finds the tumor bounding box from the segmentation mask
- Crops around it with a margin for context
- Resizes/pads to 128³ (centered)

In [9]:
# Import preprocessing functions
from med3pipe.data.prepare import (
    get_bounding_box, add_margin_to_bbox, to_binary_mask,
    find_case_dirs, find_image_files, find_segmentation_files,
    merge_images, merge_segmentations, resample_like, Sam3DPaths
)

def crop_to_roi_simple(img, lbl, target_size=128, margin=10):
    """Simplified ROI cropping function."""
    # Get bounding box from label
    lbl_arr = sitk.GetArrayFromImage(lbl)
    if lbl_arr.sum() == 0:
        raise ValueError("Empty mask - no lesion found")
    
    # Find bounding box in physical space
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(lbl)
    bbox = stats.GetBoundingBox(1)  # x_start, y_start, z_start, x_size, y_size, z_size
    
    # Extract ROI with margin
    x_min = max(0, bbox[0] - margin)
    y_min = max(0, bbox[1] - margin)
    z_min = max(0, bbox[2] - margin)
    x_max = min(lbl.GetSize()[0], bbox[0] + bbox[3] + margin)
    y_max = min(lbl.GetSize()[1], bbox[1] + bbox[4] + margin)
    z_max = min(lbl.GetSize()[2], bbox[2] + bbox[5] + margin)
    
    roi_size = [x_max - x_min, y_max - y_min, z_max - z_min]
    roi_index = [x_min, y_min, z_min]
    
    # Crop both image and label
    roi_filter = sitk.RegionOfInterestImageFilter()
    roi_filter.SetSize([int(s) for s in roi_size])
    roi_filter.SetIndex([int(i) for i in roi_index])
    img_crop = roi_filter.Execute(img)
    lbl_crop = roi_filter.Execute(lbl)
    
    # Resize if larger than target
    max_dim = max(roi_size)
    if max_dim > target_size:
        scale = target_size / max_dim
        new_size = [int(s * scale) for s in roi_size]
        
        img_crop = sitk.Resample(
            img_crop, new_size, sitk.Transform(),
            sitk.sitkLinear, img_crop.GetOrigin(),
            [s * (1/scale) for s in img_crop.GetSpacing()],
            img_crop.GetDirection(), 0.0, img_crop.GetPixelID()
        )
        lbl_crop = sitk.Resample(
            lbl_crop, new_size, sitk.Transform(),
            sitk.sitkNearestNeighbor, lbl_crop.GetOrigin(),
            [s * (1/scale) for s in lbl_crop.GetSpacing()],
            lbl_crop.GetDirection(), 0.0, lbl_crop.GetPixelID()
        )
    
    # Pad to target size (centered)
    current_size = img_crop.GetSize()
    pad_needed = [(target_size - s) for s in current_size]
    lower_pad = [p // 2 for p in pad_needed]
    upper_pad = [p - lower_pad[i] for i, p in enumerate(pad_needed)]
    
    img_final = sitk.ConstantPad(img_crop, lower_pad, upper_pad, 0.0)
    lbl_final = sitk.ConstantPad(lbl_crop, lower_pad, upper_pad, 0)
    
    return img_final, lbl_final

# Prepare ROI-cropped data
paths = Sam3DPaths(sam3d_root=SAM3D_ROOT, category=CATEGORY, ct_name=CT_NAME)
paths.ensure()

# Find and process cases
case_dirs = find_case_dirs(DATASET_ROOT)
print(f"Found {len(case_dirs)} cases")

prepared = 0
for case_dir in case_dirs:  # Process ALL cases (removed [:5] limit)
    case_id = case_dir.parent.parent.name
    
    try:
        img_files = find_image_files(case_dir)
        seg_files = find_segmentation_files(case_dir)
        
        if not img_files or not seg_files:
            print(f"[SKIP] {case_id}: Missing files")
            continue
        
        img = merge_images(img_files)
        lbl = merge_segmentations(seg_files)
        
        # Align geometries
        if img.GetSize() != lbl.GetSize():
            img = resample_like(img, reference=lbl, is_label=False)
        
        # Convert to binary mask
        lbl_bin = to_binary_mask(lbl)
        
        # ROI crop
        img_roi, lbl_roi = crop_to_roi_simple(img, lbl_bin, TARGET_SIZE, ROI_MARGIN)
        
        # Save
        out_image = paths.images_tr / f"{case_id}.nii.gz"
        out_label = paths.labels_tr / f"{case_id}.nii.gz"
        sitk.WriteImage(img_roi, str(out_image))
        sitk.WriteImage(lbl_roi, str(out_label))
        
        prepared += 1
        if prepared % 10 == 0:
            print(f"  Processed {prepared} cases...")
    except Exception as e:
        print(f"[ERROR] {case_id}: {e}")

print(f"\n✅ Prepared {prepared} ROI-cropped cases → {paths.train_root}")

Found 246 cases
  Processed 10 cases...
  Processed 20 cases...
  Processed 30 cases...
  Processed 40 cases...
  Processed 50 cases...
  Processed 60 cases...
  Processed 70 cases...
  Processed 80 cases...
  Processed 90 cases...
  Processed 100 cases...
  Processed 110 cases...
  Processed 120 cases...
  Processed 130 cases...
  Processed 140 cases...
  Processed 150 cases...
  Processed 160 cases...
  Processed 170 cases...
  Processed 180 cases...
  Processed 190 cases...
  Processed 200 cases...
  Processed 210 cases...
  Processed 220 cases...
  Processed 230 cases...
  Processed 240 cases...

✅ Prepared 246 ROI-cropped cases → c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST_roi


## 3. Load SAM-Med3D Model with medim

Load the pre-trained SAM-Med3D model using medim (simple one-line interface).

In [10]:
# Check if checkpoint exists
if not CHECKPOINT_PATH.exists():
    print(f"⚠️  Checkpoint not found: {CHECKPOINT_PATH}")
    print("   Download from: https://huggingface.co/blueyo0/SAM-Med3D/resolve/main/sam_med3d_turbo.pth")
else:
    size_mb = CHECKPOINT_PATH.stat().st_size / (1024*1024)
    print(f"✅ Checkpoint found: {CHECKPOINT_PATH} ({size_mb:.1f} MB)")

# Load SAM-Med3D model via medim
print("\n📦 Loading SAM-Med3D model via medim...")
sam_model = medim.create_model(
    "SAM-Med3D",
    pretrained=True,
    checkpoint_path=str(CHECKPOINT_PATH)
).to(DEVICE)
sam_model.eval()

print(f"✅ SAM-Med3D model loaded successfully!")
print(f"   Model device: {next(sam_model.parameters()).device}")

✅ Checkpoint found: c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth (383.5 MB)

📦 Loading SAM-Med3D model via medim...
creating model SAM-Med3D
try to load pretrained weights from c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
✅ SAM-Med3D model loaded successfully!
   Model device: cpu


## 4. Extract Embeddings from Encoder

Pass the preprocessed images through the SAM-Med3D image encoder to extract embeddings.

In [11]:
# Create preprocessing transform
def make_preprocess_transform(img_size=128):
    """Create preprocessing pipeline for SAM-Med3D."""
    return tio.Compose([
        tio.ToCanonical(),
        tio.Clamp(out_min=-150, out_max=250),  # CT soft tissue window
        tio.CropOrPad(target_shape=(img_size, img_size, img_size)),
        tio.ZNormalization(masking_method=None),
    ])

def load_and_preprocess(img_path, transform):
    """Load and preprocess a single image."""
    sitk_img = sitk.ReadImage(str(img_path))
    sitk_arr, _ = tio.data.io.sitk_to_nib(sitk_img)  # (1, D, H, W)
    subject = tio.Subject(image=tio.ScalarImage(tensor=sitk_arr))
    subject = transform(subject)
    image = subject.image.data.clone().detach()  # (1, D, H, W)
    return image.unsqueeze(0)  # (1, 1, D, H, W)

# Extract embeddings
preprocess = make_preprocess_transform(TARGET_SIZE)
embeddings_dir = SAM3D_ROOT / "features" / CATEGORY / f"{CT_NAME}_train"
embeddings_dir.mkdir(parents=True, exist_ok=True)

image_files = sorted(paths.images_tr.glob("*.nii.gz"))
print(f"Extracting embeddings for {len(image_files)} images...")

embeddings = {}
with torch.no_grad():
    for img_path in image_files:
        # Remove both .nii.gz extensions properly
        case_id = img_path.name.replace('.nii.gz', '').replace('.nii', '')
        
        # Load and preprocess
        vol = load_and_preprocess(img_path, preprocess).to(DEVICE)
        
        # Extract embedding
        emb = sam_model.image_encoder(vol)  # Shape: (1, C, D', H', W')
        
        # Save embedding
        embeddings[case_id] = emb.cpu()
        torch.save({
            "path": str(img_path),
            "embedding": emb.cpu()
        }, embeddings_dir / f"{case_id}_embedding.pt")

print(f"\n✅ Extracted {len(embeddings)} embeddings → {embeddings_dir}")
print(f"   Embedding shape example: {list(embeddings.values())[0].shape}")

Extracting embeddings for 246 images...

✅ Extracted 246 embeddings → c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_roi_train
   Embedding shape example: torch.Size([1, 384, 8, 8, 8])


## 5. Pool Embeddings to Feature Vectors

Apply Global Average Pooling (GAP) to convert 3D embeddings to 1D feature vectors.

In [12]:
def pool_embedding(emb_tensor):
    """Apply Global Average Pooling to embedding.
    
    Args:
        emb_tensor: Shape (1, C, D, H, W)
    Returns:
        Feature vector of shape (C,)
    """
    # Average over spatial dimensions (D, H, W)
    pooled = emb_tensor.mean(dim=[0, 2, 3, 4])  # Result: (C,)
    return pooled

# Pool all embeddings
case_ids = []
features = []

for case_id, emb in embeddings.items():
    pooled = pool_embedding(emb)
    case_ids.append(case_id)
    features.append(pooled.numpy())

X = np.stack(features, axis=0)  # Shape: (n_cases, n_features)

print(f"✅ Pooled embeddings to feature vectors")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Cases: {case_ids}")

✅ Pooled embeddings to feature vectors
   Feature matrix shape: (246, 384)
   Cases: ['GIST-001_CT', 'GIST-002_CT', 'GIST-003_CT', 'GIST-004_CT', 'GIST-005_CT', 'GIST-006_CT', 'GIST-007_CT', 'GIST-008_CT', 'GIST-009_CT', 'GIST-010_CT', 'GIST-011_CT', 'GIST-012_CT', 'GIST-013_CT', 'GIST-014_CT', 'GIST-015_CT', 'GIST-016_CT', 'GIST-017_CT', 'GIST-018_CT', 'GIST-019_CT', 'GIST-020_CT', 'GIST-021_CT', 'GIST-022_CT', 'GIST-023_CT', 'GIST-024_CT', 'GIST-025_CT', 'GIST-026_CT', 'GIST-027_CT', 'GIST-028_CT', 'GIST-029_CT', 'GIST-030_CT', 'GIST-031_CT', 'GIST-032_CT', 'GIST-033_CT', 'GIST-034_CT', 'GIST-035_CT', 'GIST-036_CT', 'GIST-037_CT', 'GIST-038_CT', 'GIST-039_CT', 'GIST-040_CT', 'GIST-041_CT', 'GIST-042_CT', 'GIST-043_CT', 'GIST-044_CT', 'GIST-045_CT', 'GIST-046_CT', 'GIST-047_CT', 'GIST-048_CT', 'GIST-049_CT', 'GIST-050_CT', 'GIST-051_CT', 'GIST-052_CT', 'GIST-053_CT', 'GIST-054_CT', 'GIST-055_CT', 'GIST-056_CT', 'GIST-057_CT', 'GIST-058_CT', 'GIST-059_CT', 'GIST-060_CT', 'GIST-061_CT',

## 6. Load Labels from sheet.csv

Load ground truth labels and align them with the case IDs.

In [13]:
import pandas as pd

# Load labels from sheet (located at project root)
sheet_path = project_root / "sheet.csv"
df = pd.read_csv(sheet_path)

print(f"Loaded {len(df)} rows from sheet.csv")
print(f"Available case_ids from embeddings: {case_ids[:5]}...")  # Show first 5

# Filter sheet for current dataset (e.g., only GIST rows)
df_filtered = df[df['Dataset'].str.upper() == CATEGORY.upper()].copy()
print(f"Filtered to {len(df_filtered)} rows for dataset '{CATEGORY}'")

# Create case_id to label mapping
# The Subject column has format like "GIST-001" or just the number part in the nifti filename
label_map = {}
for _, row in df_filtered.iterrows():
    subject = str(row["Subject"])
    
    # Try multiple possible case_id formats
    # Format 1: Full subject ID with _CT suffix (e.g., "GIST-001_CT")
    case_id_1 = f"{subject}_CT"
    # Format 2: Just the numeric part with _CT (e.g., "001_CT")
    if '-' in subject:
        numeric_part = subject.split('-')[-1]
        case_id_2 = f"{numeric_part}_CT"
    else:
        case_id_2 = f"{subject}_CT"
    
    # Store both formats
    label = int(row["Diagnosis_binary"])
    label_map[case_id_1] = label
    label_map[case_id_2] = label
    label_map[subject] = label  # Also store plain subject ID

print(f"Created label map with {len(label_map)} entries")
print(f"Sample label_map keys: {list(label_map.keys())[:10]}")

# Align labels with feature matrix
y = []
aligned_case_ids = []
aligned_features = []

for i, case_id in enumerate(case_ids):
    if case_id in label_map:
        aligned_case_ids.append(case_id)
        aligned_features.append(features[i])
        y.append(label_map[case_id])
    else:
        print(f"⚠️  No label found for {case_id}")

if len(aligned_features) == 0:
    print("\n❌ ERROR: No matching cases found!")
    print(f"   Case IDs from embeddings: {case_ids}")
    print(f"   Available subjects in sheet: {df_filtered['Subject'].tolist()[:10]}")
    raise ValueError("No matching cases between embeddings and labels. Check case_id format.")

X = np.stack(aligned_features, axis=0)
y = np.array(y)

print(f"\n✅ Loaded and aligned labels")
print(f"   Final dataset size: {len(y)} cases")
print(f"   Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

Loaded 930 rows from sheet.csv
Available case_ids from embeddings: ['GIST-001_CT', 'GIST-002_CT', 'GIST-003_CT', 'GIST-004_CT', 'GIST-005_CT']...
Filtered to 246 rows for dataset 'gist'
Created label map with 738 entries
Sample label_map keys: ['GIST-001_CT', '001_CT', 'GIST-001', 'GIST-002_CT', '002_CT', 'GIST-002', 'GIST-003_CT', '003_CT', 'GIST-003', 'GIST-004_CT']

✅ Loaded and aligned labels
   Final dataset size: 246 cases
   Class distribution: {np.int64(0): np.int64(121), np.int64(1): np.int64(125)}


## 7. Classification with TabPFN

Perform dimensionality reduction (PCA) and classification with TabPFN.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Check if we have enough samples
n_samples = len(y)
n_classes = len(np.unique(y))

print(f"Dataset: {n_samples} samples, {n_classes} classes")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

if n_samples < 10:
    print(f"\n⚠️  WARNING: Only {n_samples} samples available.")
    print("   For demonstration purposes, using a simple split without stratification.")
    print("   For real experiments, process more cases in step 2!\n")
    
    # For very small datasets, use a simple split
    test_size = max(2, n_samples // 5)  # At least 2 samples in test
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=42, shuffle=True
    )
else:
    # Normal stratified split for larger datasets
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

print(f"Train set: {len(y_train)} samples")
print(f"Val set: {len(y_val)} samples")

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# PCA (TabPFN works best with ≤500 features)
n_components = min(500, X_train_scaled.shape[0] - 1, X_train_scaled.shape[1])
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print(f"\n✅ Preprocessing complete")
print(f"   PCA components: {n_components}")
print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# Train TabPFN
print("\n📊 Training TabPFN...")

# Add TabPFN to path if needed
tabpfn_src = project_root / "TabPFN-main" / "TabPFN-main" / "src"
if tabpfn_src.exists() and str(tabpfn_src) not in sys.path:
    sys.path.insert(0, str(tabpfn_src))

from tabpfn.classifier import TabPFNClassifier

clf = TabPFNClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
clf.fit(X_train_pca, y_train)

# Evaluate
y_pred = clf.predict(X_val_pca)
y_proba = clf.predict_proba(X_val_pca)

accuracy = accuracy_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_proba[:, 1]) if len(np.unique(y)) == 2 else None

print(f"\n✅ TabPFN Classification Results")
print(f"   Accuracy: {accuracy:.3f}")
if roc_auc:
    print(f"   ROC AUC: {roc_auc:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred))

Dataset: 246 samples, 2 classes
Class distribution: {np.int64(0): np.int64(121), np.int64(1): np.int64(125)}
Train set: 196 samples
Val set: 50 samples

✅ Preprocessing complete
   PCA components: 195
   Explained variance: 1.000

📊 Training TabPFN...


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)



✅ TabPFN Classification Results
   Accuracy: 0.620
   ROC AUC: 0.696

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.48      0.56        25
           1       0.59      0.76      0.67        25

    accuracy                           0.62        50
   macro avg       0.63      0.62      0.61        50
weighted avg       0.63      0.62      0.61        50



## Summary

This notebook demonstrates a self-sufficient pipeline:
1. ✅ **ROI-cropped preprocessing** - Tumor-centered volumes preserving lesion resolution
2. ✅ **SAM-Med3D loading** - Via medim (one-line interface)
3. ✅ **Embedding extraction** - Direct encoder usage
4. ✅ **Feature pooling** - Global Average Pooling
5. ✅ **Label loading** - From sheet.csv
6. ✅ **TabPFN classification** - With PCA preprocessing

The pipeline is minimal and self-contained, using only essential functions from the codebase.